<a href="https://colab.research.google.com/github/sabdaaf/analisis-sentimen-twitter-svm-dt-indobert/blob/main/sentimen_analisis_x_NB_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ANALISIS SENTIMEN #KABURAJADULU TWITTER/X**
===========================================
Pipeline : Cleaning -> Labeling (InSet Lexicon) -> Feature Extraction (TF-IDF)
           -> Training (SVM & Naive Bayes) -> Evaluasi & Perbandingan

Perbaikan dari versi awal:
  1. URL InSet diperbaiki (branch `master`, file di root repo, bukan /lexicon/).
  2. positive.tsv/negative.tsv punya header -> dibaca dengan header=0 supaya
     kolom `weight` bertipe numerik (versi lama: TypeError saat weight > 0).
  3. 1.142 kata beririsan di kedua file lexicon: bobotnya DIJUMLAHKAN, bukan
     ditimpa (versi lama membuat 'ramah' +5 menjadi -3).
  4. DATA LEAKAGE dihilangkan: TF-IDF di-fit HANYA pada data train.
  5. Labeling di-vektorisasi (tanpa iterrows) + memakai bobot InSet.
  6. Normalisasi slang pakai kamus-alay (~3.5k entri), fallback ke kamus manual.
  7. Hasil stemming Sastrawi di-cache ke disk (proses ini lambat sekali).

Dependensi:  pip install Sastrawi pandas scikit-learn matplotlib seaborn openpyxl requests
Jalankan  :  python sentimen_kaburajadulu.py


In [44]:
import io
import os
import re
import warnings

import matplotlib
matplotlib.use("Agg")           # aman untuk environment tanpa display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

warnings.filterwarnings("ignore", category=UserWarning)

Config

In [45]:
CONFIG = {
    "input_path": "scrapingTwitter.xlsx",
    "text_col": "text",
    "id_col": "id",
    "random_state": 42,
    "test_size": 0.2,
    "max_features": 5000,
    "neutral_threshold": 0.0,  # |skor| <= threshold dianggap netral
    "stem_cache": "stemmed_cache.csv",
}

# URL yang BENAR (branch master, file di root repo)
POS_URL = "https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv"
NEG_URL = "https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv"
SLANG_URL = ("https://raw.githubusercontent.com/nasalsabila/kamus-alay/"
             "master/colloquial-indonesian-lexicon.csv")

# Fallback kalau kamus-alay tidak bisa diunduh
SLANG_FALLBACK = {
    "bgt": "banget", "yg": "yang", "gw": "saya", "gua": "saya",
    "lu": "kamu", "pake": "pakai", "krn": "karena", "gk": "tidak",
    "ga": "tidak", "gak": "tidak", "sampe": "sampai", "tp": "tapi",
    "dr": "dari", "utk": "untuk", "sm": "sama", "jd": "jadi",
    "udah": "sudah", "udh": "sudah", "blm": "belum", "dgn": "dengan",
}

**load data**

In [46]:
def load_data(path):
    df = pd.read_excel(path, engine="openpyxl")
    print(f"Data awal: {df.shape[0]} baris, {df.shape[1]} kolom")
    return df

**TEXT CLEANING**

In [48]:
def clean_noise(text):
    """Hapus URL, mention, simbol hashtag, newline, emoji, angka, spasi ganda."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text, flags=re.MULTILINE)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"[\r\n]+", " ", text)
    text = re.sub(r"[^\w\s,.!?]", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_slang_dict():
    """Unduh kamus-alay (~3.5k entri). Fallback ke kamus manual jika gagal."""
    try:
        r = requests.get(SLANG_URL, timeout=30)
        r.raise_for_status()
        alay = pd.read_csv(io.StringIO(r.text))
        alay = alay.dropna(subset=["slang", "formal"])
        d = dict(zip(alay["slang"].astype(str).str.lower(),
                     alay["formal"].astype(str).str.lower()))
        d.update(SLANG_FALLBACK)          # kamus manual menang bila bentrok
        print(f"Kamus slang dimuat: {len(d)} entri (kamus-alay)")
        return d
    except Exception as e:
        print(f"Gagal memuat kamus-alay ({e}). Pakai kamus fallback "
              f"({len(SLANG_FALLBACK)} entri).")
        return dict(SLANG_FALLBACK)


def preprocess_base(df, text_col, slang_dict):
    """Cleaning + case folding + normalisasi slang + buang kosong/duplikat."""
    df = df[[CONFIG["id_col"], text_col]].copy()
    df["text_clean"] = (df[text_col].apply(clean_noise)
                                    .str.lower()
                                    .apply(lambda t: " ".join(
                                        slang_dict.get(w, w) for w in t.split())))

    before = len(df)
    df = df[df["text_clean"].str.strip().str.len() > 2].reset_index(drop=True)
    df = df.drop_duplicates(subset="text_clean").reset_index(drop=True)
    print(f"Baris dibuang (kosong/terlalu pendek/duplikat): {before - len(df)}")
    print(f"Ukuran data setelah preprocessing dasar: {df.shape}")
    return df


** LEXICON-BASED LABELING**

In [49]:
def load_lexicon(pos_url=POS_URL, neg_url=NEG_URL):
    """Muat InSet menjadi SATU kamus bobot bersih.

    Catatan penting: ada ~1.142 kata yang muncul di positive.tsv DAN
    negative.tsv. Memakai dict.update() dua kali (seperti versi lama) membuat
    bobot negatif menimpa bobot positif -- kata 'ramah' (+5) jadi -3 dan
    'senang' (+5) jadi -4, sehingga kalimat positif ter-label negatif.
    Solusinya: bobot kedua file DIJUMLAHKAN -> 'ramah' = +5 + (-3) = +2.
    """
    parts = {}
    for name, url in [("positif", pos_url), ("negatif", neg_url)]:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            d = pd.read_csv(io.StringIO(r.text), sep="\t", header=0)
            d.columns = ["word", "weight"]
            d["weight"] = pd.to_numeric(d["weight"], errors="coerce")
            d = d.dropna(subset=["word", "weight"])
            d["word"] = d["word"].astype(str).str.strip().str.lower()
            d = d[d["word"] != ""]
            parts[name] = dict(zip(d["word"], d["weight"].astype(float)))
            print(f"Lexicon {name}: {len(d)} kata dimuat.")
        except Exception as e:
            print(f"Gagal memuat lexicon {name}: {e}")
            return {}

    pos, neg = parts["positif"], parts["negatif"]
    overlap = set(pos) & set(neg)
    lexicon = {w: pos.get(w, 0.0) + neg.get(w, 0.0) for w in set(pos) | set(neg)}
    lexicon = {w: v for w, v in lexicon.items() if v != 0.0}
    print(f"Kata beririsan pos & neg: {len(overlap)} (bobot dijumlahkan)")
    print(f"Total entri lexicon efektif: {len(lexicon)}")
    return lexicon


def apply_lexicon_labeling(df, lexicon, threshold=0.0):
    """Skor = jumlah bobot InSet tiap kata. Vektorisasi, tanpa iterrows."""
    if "text_clean" not in df.columns:
        raise ValueError("DataFrame harus punya kolom 'text_clean'.")

    df = df.copy()
    get = lexicon.get
    df["sentiment_score"] = df["text_clean"].map(
        lambda t: float(sum(get(w, 0.0) for w in t.split()))
    )
    df["n_lexicon_hit"] = df["text_clean"].map(
        lambda t: sum(1 for w in t.split() if w in lexicon)
    )
    df["label"] = np.select(
        [df["sentiment_score"] > threshold, df["sentiment_score"] < -threshold],
        ["positive", "negative"],
        default="neutral",
    )

    print("\nDistribusi label:")
    print(df["label"].value_counts().to_string())
    print((df["label"].value_counts(normalize=True) * 100).round(2)
          .to_string(header=False))
    no_hit = (df["n_lexicon_hit"] == 0).mean() * 100
    print(f"Tweet tanpa satu pun kata lexicon: {no_hit:.1f}% "
          f"(semuanya otomatis jadi 'neutral')")
    if no_hit > 50:
        print("PERINGATAN: mayoritas tweet tidak ter-cover lexicon. "
              "Pertimbangkan perluas normalisasi slang atau labeli manual "
              "sebagian sebagai gold standard.")
    return df

**FITUR: TF-IDF**

In [50]:
_stopword_remover = StopWordRemoverFactory().create_stop_word_remover()
_stemmer = StemmerFactory().create_stemmer()


def preprocess_for_tfidf(df):
    """Stopword removal + stemming. Hasil di-cache karena Sastrawi lambat."""
    cache = CONFIG["stem_cache"]
    if os.path.exists(cache):
        cached = pd.read_csv(cache)
        if len(cached) == len(df):
            print(f"Memakai cache stemming: {cache}")
            df = df.copy()
            df["text_tfidf_ready"] = cached["text_tfidf_ready"].fillna("").values
            return df

    print("Stemming + stopword removal (bisa beberapa menit)...")
    df = df.copy()
    df["text_tfidf_ready"] = df["text_clean"].map(
        lambda t: _stemmer.stem(_stopword_remover.remove(t))
    )
    df[["text_tfidf_ready"]].to_csv(cache, index=False)
    print(f"Hasil stemming disimpan ke {cache}")
    return df


def build_tfidf_train_test(train_texts, test_texts, max_features=5000):
    """PENTING: fit HANYA di train, transform di test -> tanpa data leakage."""
    vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2),
                          min_df=2, sublinear_tf=True)
    X_train = vec.fit_transform(train_texts)
    X_test = vec.transform(test_texts)
    print(f"Dimensi TF-IDF -> train: {X_train.shape}, test: {X_test.shape}")
    return X_train, X_test, vec

**TRAINING & TUNING **

In [51]:
def train_svm(X_train, y_train, cv=5):
    grid = GridSearchCV(
        SVC(class_weight="balanced", random_state=CONFIG["random_state"]),
        {"C": [0.1, 1, 10], "kernel": ["linear", "rbf"]},
        cv=StratifiedKFold(cv, shuffle=True, random_state=CONFIG["random_state"]),
        scoring="f1_macro", n_jobs=-1,
    )
    grid.fit(X_train, y_train)
    print(f"Best SVM params: {grid.best_params_}")
    return grid.best_estimator_


def train_naive_bayes(X_train, y_train, cv=5):
    """MultinomialNB cocok untuk TF-IDF (nilai non-negatif)."""
    grid = GridSearchCV(
        MultinomialNB(),
        {"alpha": [0.1, 0.5, 1.0, 2.0], "fit_prior": [True, False]},
        cv=StratifiedKFold(cv, shuffle=True, random_state=CONFIG["random_state"]),
        scoring="f1_macro", n_jobs=-1,
    )
    grid.fit(X_train, y_train)
    print(f"Best Naive Bayes params: {grid.best_params_}")
    return grid.best_estimator_


**EVALUASI **

In [52]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1m = f1_score(y_test, y_pred, average="macro")

    print(f"\n=== {model_name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1-macro : {f1m:.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))

    labels = sorted(pd.unique(y_test))
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Prediksi")
    plt.ylabel("Aktual")
    plt.tight_layout()
    plt.savefig(f"confmat_{model_name.replace(' ', '_')}.png", dpi=120)
    plt.close()

    return {"model": model_name, "accuracy": acc, "f1_macro": f1m}

**MAIN PIPELINE **

In [ ]:
def main():
    # --- Tahap 1-3: load + preprocessing dasar ---
    df = preprocess_base(load_data(CONFIG["input_path"]),
                         CONFIG["text_col"], load_slang_dict())

    # --- Tahap 4: labeling lexicon ---
    lexicon = load_lexicon()
    if not lexicon:
        print("Lexicon gagal dimuat. Pipeline dihentikan.")
        return df, pd.DataFrame()
    df = apply_lexicon_labeling(df, lexicon, CONFIG["neutral_threshold"])
    df.to_csv("labeled_data_for_review.csv", index=False)
    print("\nData berlabel disimpan: labeled_data_for_review.csv "
          "(disarankan cek/koreksi manual sebelum training)\n")

    if df["label"].nunique() < 2:
        print("Hanya ada satu kelas. Tidak bisa training.")
        return df, pd.DataFrame()

    y = df["label"].values

    # --- Tahap 5: SPLIT DULU, baru ekstraksi fitur (anti-leakage) ---
    idx_train, idx_test = train_test_split(
        np.arange(len(df)), test_size=CONFIG["test_size"],
        random_state=CONFIG["random_state"], stratify=y)
    y_train, y_test = y[idx_train], y[idx_test]
    print(f"Split -> train: {len(idx_train)}, test: {len(idx_test)}")

    # --- Tahap 6: fitur TF-IDF ---
    df = preprocess_for_tfidf(df)
    texts = df["text_tfidf_ready"].fillna("").values
    X_train, X_test, _ = build_tfidf_train_test(
        texts[idx_train], texts[idx_test], CONFIG["max_features"])

    # --- Tahap 7-8: training & evaluasi ---
    results = [
        evaluate_model(train_svm(X_train, y_train), X_test, y_test, "SVM_TFIDF"),
        evaluate_model(train_naive_bayes(X_train, y_train), X_test, y_test,
                       "NaiveBayes_TFIDF"),
    ]

    # --- Tahap 9: ringkasan perbandingan ---
    results_df = (pd.DataFrame(results)
                    .sort_values("f1_macro", ascending=False)
                    .reset_index(drop=True))
    print("\n=== RINGKASAN PERBANDINGAN MODEL ===")
    print(results_df.to_string(index=False))
    results_df.to_csv("model_comparison_results.csv", index=False)

    return df, results_df


if __name__ == "__main__":
    df_final, results_df = main()

Data awal: 11940 baris, 32 kolom
Kamus slang dimuat: 4333 entri (kamus-alay)
Baris dibuang (kosong/terlalu pendek/duplikat): 531
Ukuran data setelah preprocessing dasar: (11409, 3)
Lexicon positif: 3609 kata dimuat.
Lexicon negatif: 6609 kata dimuat.
Kata beririsan pos & neg: 1142 (bobot dijumlahkan)
Total entri lexicon efektif: 8853

Distribusi label:
label
negative    9465
positive    1244
neutral      700
negative    82.96
positive    10.90
neutral      6.14
Tweet tanpa satu pun kata lexicon: 4.3% (semuanya otomatis jadi 'neutral')

Data berlabel disimpan: labeled_data_for_review.csv (disarankan cek/koreksi manual sebelum training)

Split -> train: 9127, test: 2282
Stemming + stopword removal (bisa beberapa menit)...
